# Pre-Deployment Antagonism Discriminator — Subsampling Validation (Read-Only Summary)

**Motivation**: The paper should not rest on the slogan "apply post-hoc correction by default" — the conditional
statement help ⟺ ρ > σ_c/(2σ_r) (exp_r26) gives the theoretical criterion for when multiplicative correction helps.
This experiment turns that criterion into a **pre-deployment diagnostic** and quantifies its sample requirement:
given only k measured substations, a deployer estimates (ρ̂, σ̂_r) from those k stations (σ_c can always be computed
exactly, since the allocation output is the deployer's own product), and the discriminator outputs help/hurt. How
well does that agreement rate track the ground-truth verdict from the full station set (the sign of ΔRMSE)?

## Protocol at a Glance

- **Combined universe**: UK 240 (exp_r25) + AU 180 (au_phase_points) — substation-level vectors are reconstructed
  following the 024/011 function patterns and anchored to frozen values per combination (max deviation ~1e-15)
- **k tiers**: {3, 5, 8, 10, 15, 20} ∪ {full}; truncated when k ≥ number of stations in a region
  (UK median station count 101, minimum 50 → all tiers usable; AU median 12, minimum 5 → coverage drops at high k)
- **Per (combination, k)**: draw k stations without replacement x 500 repetitions (SeedSequence fixed, order-independent)
- **Degeneracy guard**: zero variance in log F or log ρ across the k stations -> conservatively output hurt (0.07% of cases, all in AU's small-k tiers)
- **Second tier (no ground truth)**: a logistic discriminator on σ_c + base-curve type + base-allocation entropy — quantifies how far one can get without measurements

Generating script: `034_exp_deploy_diagnostic.py`; this notebook only reads `results/exp_deploy_diag/`.

In [ ]:
# Load artifacts from disk (read-only)
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUT = Path('..') / 'results' / 'exp_deploy_diag'
sub = pd.read_csv(OUT / 'subsample_accuracy.csv',
                  dtype={'seed': str, 'fold': str, 'k_label': str})
curves = pd.read_csv(OUT / 'k_accuracy_curves.csv', dtype={'k_label': str})
min_k = pd.read_csv(OUT / 'min_k_table.csv', dtype={'min_k': str})
with open(OUT / 'truthfree_baseline.json', encoding='utf-8') as f:
    truthfree = json.load(f)
with open(OUT / 'diagnostic_summary.json', encoding='utf-8') as f:
    summary = json.load(f)

print(f"Detail {len(sub)} rows | Curves {len(curves)} rows | min_k {len(min_k)} rows")
print(f"Anchor check: UK max deviation {summary['anchor_check']['uk']['max_abs_dev']:.2e} | "
      f"AU {summary['anchor_check']['au']['max_abs_dev']:.2e} | "
      f"UK GNN entropy {summary['anchor_check']['uk_gnn_entropy']['max_abs_dev']:.2e}")
for name, v in summary['verdicts'].items():
    print(f"{name}: value={v['value']:.4f} → {v['verdict']}")

## 1. Family of k-Accuracy Curves (per case x base-curve group)

- Dashed lines = 80% / 90% target agreement rate; the rightmost x-axis tier = the full station set (= the accuracy
  of the phase-diagram theoretical criterion itself, i.e. the discriminator's **information ceiling**).
- **AU's high-k tiers have coverage < 1** (truncated when k ≥ the number of stations in a region; coverage is
  annotated on the points) — readings at k=15/20 only represent the small number of regions with enough stations,
  and must be interpreted together with the coverage value.

In [ ]:
K_ORDER = ['3', '5', '8', '10', '15', '20', 'full']
GROUP_STYLE = {
    'all': dict(color='#1f77b4', lw=2.5, marker='o', label='All base curves'),
    'static': dict(color='#2ca02c', lw=1.8, marker='s', label='Static arms (Uni+GPM)'),
    'gnn': dict(color='#d62728', lw=1.8, marker='^', label='GNN arm'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, case in zip(axes, ('UK', 'AU')):
    for group, style in GROUP_STYLE.items():
        g = curves[(curves['case'] == case) & (curves['group'] == group)]
        g = g.set_index('k_label').reindex(K_ORDER).dropna(subset=['accuracy'])
        xs = [K_ORDER.index(k) for k in g.index]
        ax.plot(xs, g['accuracy'], **style)
        # Annotate points where coverage < 1 (AU truncated tiers)
        for x, (_, row) in zip(xs, g.iterrows()):
            if row['coverage'] < 0.999:
                ax.annotate(f"cov {row['coverage']:.0%}", (x, row['accuracy']),
                            textcoords='offset points', xytext=(0, -14),
                            fontsize=7, ha='center', color='gray')
    for t, c in ((0.8, '0.55'), (0.9, '0.35')):
        ax.axhline(t, color=c, ls='--', lw=0.8)
        ax.text(-0.35, t + 0.005, f'{t:.0%}', fontsize=8, color=c)
    ax.set_xticks(range(len(K_ORDER)))
    ax.set_xticklabels(['3', '5', '8', '10', '15', '20', 'Full'])
    ax.set_xlabel('Number of measured substations k')
    ax.set_title(f'{case} (regional median station count '
                 f"{'101' if case == 'UK' else '12'})")
    ax.grid(alpha=0.25)
axes[0].set_ylabel('Discriminator agreement rate (vs. sign of full-set ΔRMSE)')
axes[0].set_ylim(0.5, 1.02)
axes[0].legend(loc='lower right', fontsize=9)
fig.suptitle('k-Station Discriminator Agreement Rate Curve Family', y=1.02)
plt.tight_layout()
plt.show()

**How to read this**:
- **AU**: pooling across all base curves, k=5 already reaches 80% and k=8 reaches 90% (coverage 83%/58%
  respectively); even the region with the fewest stations (5 stations) reaches 0.77-0.88 agreement at k=3 —
  **the discriminator holds up under AU's small-sample conditions**.
- **UK static arms**: k=5 reaches 80%, k=8 reaches 90% — the same sample efficiency as AU.
- **The UK GNN arm is the only failure surface**: the curve plateaus at 0.72 — this is not a sampling problem,
  but rather the theoretical criterion itself only achieves 72% accuracy on the UK GNN combination (the full-station
  tier = the prediction_correct fraction from the phase diagram). Adding more measured stations cannot break
  through this information ceiling.

## 2. Minimum k Required to Reach 80% / 90% Agreement

In [ ]:
tbl = min_k.copy()
tbl['target'] = (tbl['target'] * 100).astype(int).astype(str) + '%'
tbl['accuracy_at_min_k'] = tbl['accuracy_at_min_k'].round(3)
tbl['coverage_at_min_k'] = tbl['coverage_at_min_k'].round(2)
tbl['max_accuracy_on_curve'] = tbl['max_accuracy_on_curve'].round(3)
tbl = tbl.rename(columns={
    'case': 'Case', 'group': 'Base Group', 'target': 'Target', 'min_k': 'Min k',
    'accuracy_at_min_k': 'Accuracy at Min k', 'coverage_at_min_k': 'Coverage at Min k',
    'max_accuracy_on_curve': 'Max Accuracy on Curve'})
pivot_view = tbl.pivot_table(index=['Case', 'Base Group'], columns='Target',
                             values='Min k', aggfunc='first')
display(pivot_view)
display(tbl)

**Headline**: UK static arms 80%/90% → k=5/8; AU (all base curves) → k=5/8;
UK all-base-curves pooled needs the full-station tier to reach 80% (dragged down by the GNN arm), and 90% is never
reached (ceiling 83.3%); the UK GNN arm reaches neither target (ceiling 72.2%).

> `min_k` rule = first crossing on the pooled curve (the first tier, in ascending k, where accuracy ≥ target);
> AU's k=8/10 tiers have 58% coverage — the minimum-k reading represents "regions with enough stations to draw k from".

## 3. AU Small-Sample Focus: Per-Region Agreement Rate (where the discriminator's practical value is decided)

Each AU region has only 5-25 measured substations (median 12). The table below shows the average agreement rate,
per region, across the 15 combinations (5 base curves x 3 signals) for each k tier.

In [ ]:
au = sub[sub['case'] == 'AU']
piv = au.pivot_table(index='location', columns='k_label', values='accuracy')
piv = piv[[c for c in K_ORDER if c in piv.columns]]
piv.insert(0, 'n_stations', au.groupby('location')['n_substations'].first())
piv = piv.sort_values('n_stations')
display(piv.round(3))
worst3 = piv['3'].min()
print(f"Worst regional agreement rate at k=3 tier: {worst3:.3f} ({piv['3'].idxmin()}); "
      f"5-station regions (Ryde/Parramatta) at k=3 tier = "
      f"{piv.loc['Sydney_Ryde', '3']:.3f} / {piv.loc['Sydney_Parramatta', '3']:.3f}")

## 4. Two-Tier Discriminator Comparison: Slogan Baseline vs. No-Ground-Truth Heuristic vs. k-Station Discriminator

- **Slogan baseline (always-apply)** = the blanket recommendation to "apply post-hoc correction by default":
  accuracy = the actual fraction of combinations where correction helps.
- **No-ground-truth heuristic** = a logistic discriminator using only observable quantities (σ_c, base-curve type,
  base-allocation entropy), reported via leave-one-region-out cross-validation (in-sample accuracy = an upper-bound
  reading, see JSON).
- **k-station discriminator** = the main subject of this experiment (k=5 / k=10 / full-station tiers).

In [ ]:
bars = {}
for case in ('UK', 'AU'):
    tb = truthfree['truth_based_reference'][case]
    bars[case] = {
        'Slogan baseline\n(always-apply)': truthfree['slogan_baselines'][case]['always_apply_accuracy'],
        'No-ground-truth logistic\n(LOGO-CV, within-case)': truthfree['models_per_case'][case]['per_case'][case]['logo_cv_accuracy'],
        'k=5 discriminator': tb['k_5']['accuracy'],
        'k=10 discriminator': tb['k_10']['accuracy'],
        'Full-station discriminator\n(theoretical ceiling)': tb['k_full']['accuracy'],
    }

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
colors = ['#9e9e9e', '#ff7f0e', '#87b5d8', '#4292c6', '#08519c']
for ax, case in zip(axes, ('UK', 'AU')):
    names = list(bars[case])
    vals = [bars[case][n] for n in names]
    ax.bar(range(len(names)), vals, color=colors, width=0.62)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.008, f'{v:.3f}', ha='center', fontsize=9)
    ax.axhline(0.8, color='0.55', ls='--', lw=0.8)
    ax.axhline(0.9, color='0.35', ls='--', lw=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=8.5)
    ax.set_title(case)
    ax.set_ylim(0.5, 1.05)
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('Discriminator accuracy')
fig.suptitle('Two-Tier Discriminator Comparison: No Measurement (left two tiers) vs. Measured k Stations (right three tiers)', y=1.02)
plt.tight_layout()
plt.show()

mp = truthfree['models_pooled']
print(f"No-ground-truth feature increment (pooled LOGO): σ_c only "
      f"{mp['sigma_c_only']['logo_cv_accuracy']:.3f} → +base-curve type "
      f"{mp['sigma_c_basetype']['logo_cv_accuracy']:.3f} → +entropy "
      f"{mp['full_observable']['logo_cv_accuracy']:.3f}"
      f" (in-sample ceiling {mp['full_observable']['in_sample_accuracy']:.3f})")

**How to read this (how far each tier can get)**:
- **UK**: slogan baseline 0.629 → no-ground-truth logistic 0.808 → k=5 measured 0.708 → k=10 measured
  0.752 → full-station 0.833. The no-ground-truth heuristic **outperforms small-k measured discrimination** on UK
  (it implicitly learns the structural rule "GNN base curve + high σ_c → hurt", whereas small-k ρ̂ estimates are too
  noisy); but both tiers are capped by the theoretical criterion's 83% ceiling.
- **AU**: the slogan baseline itself is already 0.933 (93% of combinations do benefit from correction) — the
  no-ground-truth logistic (0.933) **does not beat the slogan**, while the measured discriminator at k≥8 (0.92+)
  can identify most of that 7% of combinations where correction hurts.
- **Overall structure**: the no-ground-truth tier can raise the "should we run correction" decision from the slogan
  to ~0.79 (pooled); crossing 90% requires actual measurement, and 8-10 stations are enough (for the static arms
  and AU); the bottleneck for the UK GNN arm lies in the theoretical criterion itself, not in sample size.

## 5. Conclusion Transcript (generated programmatically from the numbers; see diagnostic_summary.json)

In [ ]:
for name, v in summary['verdicts'].items():
    print(f"{name}:")
    print(f"  {v['description']}")
    print(f"  value = {v['value']:.4f} ({v['value_rule']}) → **{v['verdict']}**")
print()
print('k-tier truncation registry (source of AU high-k coverage):')
reg = summary['k_truncation_registry']
au_reg = {k.split('|')[1]: v for k, v in reg.items() if k.startswith('AU')}
for loc, info in sorted(au_reg.items(), key=lambda kv: kv[1]['n_substations']):
    print(f"  {loc}: {info['n_substations']} stations, available k = {info['available_ks']}, "
          f"truncated {info['truncated_ks']}")

---

**Protocol notes**:
1. σ_c can always be computed exactly and does not need subsampling — d_base/d_corr are the deployer's own
   allocation outputs and do not depend on measured demand; ε = regional total demand x 1e-6 follows the same
   logic (an input to the allocation task).
2. The full-station tier = a transcript-level recomputation of the phase-diagram rule (with a built-in consistency
   assertion, 0 violations); its aggregate value = the prediction_correct fraction from exp_r26/014 (UK 0.833 /
   AU 0.983).
3. Degenerate subsamples (zero variance across the k stations, mostly from empty-Voronoi stations sharing
   identical values) → conservatively output hurt, 860 / 1,229,500 repetitions in total (0.07%), all in AU's
   k ≤ 10 tiers.
4. Entropy feature for the no-ground-truth tier: UK GNN reads the frozen `concentration_metrics.csv` artifact
   (the kappa=1 tier, reconstruction-anchored to ≤ 1e-9); UK static and AU are all computed on the fly from the
   reconstructed base curves (same formula as script 028).